# Developing chicken heart

This notebook shows the complete `chicken_heart` analysis: data preparation,
model training, downstream analysis, and the commands used for the paper
figures. Edit the paths in **Setup** before starting a run.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
DATASET_CONFIG = 'chicken_heart'
RAW_H5AD = Path("data/chicken_heart_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/chicken_heart")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'chicken_heart_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


import CytoBridge as cb

RAW_10X_DIR = Path("data/GSE149457_RAW")
METADATA_H5AD = Path("data/chicken_heart_spatial_merged_with_meta.h5ad")
REFERENCE_ALIGNMENT_H5AD = Path("data/heart_aligned_all_timepoints.h5ad")
PREPARATION_DIR = RAW_H5AD.parent / "chicken_heart_preparation"
RUN_RAW_DATA_ASSEMBLY = False


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, config_source = load_workflow_config(DATASET_CONFIG)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "configuration",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            config_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,Developing chicken heart
1,configuration,example configuration: chicken_heart
2,raw time column,timepoint
3,cell annotation,celltype_prediction
4,observed training times,"0.0, 1.0, 2.0, 3.0"
5,classifier neighbors,1


## Data preparation

The dataset configuration records the count layer, time mapping, spatial
coordinates, and alignment settings. The command below reads the raw H5AD and
writes the aligned H5AD and edge model used for training.

### 1. preprocess
```text
cytobridge workflow --config chicken_heart --step preprocess --input-h5ad <raw.h5ad> --output-dir <run>
```

Start with: `raw H5AD and the dataset configuration`

Writes: `<run>/preprocess/chicken_heart_aligned.h5ad; <run>/preprocess/edge_classifier/chicken_heart_edge_model.pt; preprocessing records`

Next: `training`

### Assemble the chicken-heart H5AD

The downloaded 10x matrices are first matched to the reference spot roster.
The second call writes the `spatial_ot_input` coordinates used by alignment.

In [3]:
if RUN_RAW_DATA_ASSEMBLY:
    PREPARATION_DIR.mkdir(parents=True, exist_ok=True)
    reference_input = PREPARATION_DIR / "chicken_heart_reference_input.h5ad"
    cb.pp.prepare_chicken_heart_input(
        raw_dir=RAW_10X_DIR,
        metadata_h5ad=METADATA_H5AD,
        aligned_reference_h5ad=REFERENCE_ALIGNMENT_H5AD,
        output_h5ad=reference_input,
        output_table=PREPARATION_DIR / "model_input.csv",
        manifest_path=PREPARATION_DIR / "preparation.json",
        graph_database=cb.pp.bundled_graph_database_path(DATASET_CONFIG),
        repair_legacy_d7_left_right=False,
    )
    cb.pp.prepare_chicken_heart_ot_input(
        input_h5ad=reference_input,
        output_h5ad=RAW_H5AD,
        output_table=PREPARATION_DIR / "chicken_heart_ot_input.csv",
        manifest_path=PREPARATION_DIR / "ot_input.json",
    )
else:
    print("Raw-data assembly is off. Set RUN_RAW_DATA_ASSEMBLY = True to run it.")

Raw-data assembly is off. Set RUN_RAW_DATA_ASSEMBLY = True to run it.


In [4]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=config_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: Developing chicken heart (chicken_heart)
config: example configuration: chicken_heart
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/chicken_heart/preprocess/chicken_heart_aligned.h5ad
    note: Use the raw-coordinate chicken-heart input with obsm['spatial_ot_input']. D7 is pre-oriented by a recorded 180-degree rotation around its raw-stage centroid, after which the package fits expression-guided OT alignment and a fresh edge predictor.
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for training)
  downstream: skipped (GPU recommended)


In [5]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full run starts from the raw H5AD, writes the aligned data, fits the
interaction edge model when needed, and trains CytoBridge. Training requires a
CUDA-capable environment.

### 1. preprocess and train
```text
cytobridge workflow --config chicken_heart --step preprocess --step train --train --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

Start with: `raw H5AD, dataset configuration, and LR database`

Writes: `<run>/training/<stage>/best_model.pth or score_model.pth; <run>/training/adata.h5ad; training_history.csv; training_run_summary.json`

Next: `downstream`

In [6]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=config_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: Developing chicken heart (chicken_heart)
config: example configuration: chicken_heart
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/chicken_heart/preprocess/chicken_heart_aligned.h5ad
    note: Use the raw-coordinate chicken-heart input with obsm['spatial_ot_input']. D7 is pre-oriented by a recorded 180-degree rotation around its raw-stage centroid, after which the package fits expression-guided OT alignment and a fresh edge predictor.
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.human.csv
      database source: included CellChatDB resource
      interaction cutoff: 0.21681429373719752
      decision threshold source: validation-selected during de novo training
      output: tutorial_outputs/chicken_heart/preprocess/edge_classifier/chicken_hea

In [7]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis reads the aligned H5AD and fitted model from the training
directory. It writes generated states, velocity, growth, composition,
communication, ligand–receptor tables, and standard figures.

### 1. downstream
```text
cytobridge workflow --config chicken_heart --step downstream --aligned-h5ad <run>/preprocess/chicken_heart_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Start with: `aligned H5AD; <run>/training; dataset-matched LR database`

Writes: `<run>/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures`

Next: `paper-specific continuation shown in the paper-figure notebook`

In [8]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=config_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: Developing chicken heart (chicken_heart)
config: example configuration: chicken_heart
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/chicken_heart/downstream
    generated states: observed times=[0.0, 1.0, 2.0, 3.0], additional times=[0.5, 1.5, 2.5]
      simulation settings: dt=0.05, sigma=0.03, daughter noise=0, growth alpha=1
      each additional time starts from the preceding observed time point
    interpolation and classification: enabled
    time-slice velocity: enabled
    growth: enabled when present in the model
    cell-type composition: enabled
    sparse communication: enabled
    standard figures: enabled
      note: snapshots, mosaic, growth, compositio

In [9]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing aligned data or model directory: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

Continue with these commands to calculate the values used in the paper. Each
step states which downstream files it reads and which paper notebook uses its
output.

- [LR-prior ablation and stVCR comparison](../paper_figures/lr_prior_ablation_stvcr.ipynb)
- [Five-dataset benchmark](../paper_figures/loto_benchmark.ipynb)
- [Training histories](../paper_figures/training_histories.ipynb)

### 1. run the chicken-heart analyses (Main Figure 3; archived downstream bank)
```text
python scripts/run_chicken_heart_paper_downstream.py --run-root <run> --input-h5ad <run>/preprocess/chicken_heart_aligned.h5ad --model-dir <run>/training --standard-downstream <run>/downstream --output-dir <paper-run> --device cuda
```

Start with: `aligned H5AD, retrained six-stage model, and standard downstream directory`

Writes: `perturbation, interaction-off, LR-time-course, and communication-attention tables and figures`

Next: `assemble the selected Main Figure 3 panels`

The calculations are included; the updated Main Figure 3 page still needs to be assembled from them.

### 2. calculate alignment and alignment-perturbation diagnostics (S7-S8)
```text
python scripts/plot_chicken_heart_alignment.py --input-h5ad <run>/preprocess/chicken_heart_aligned.h5ad --alignment-record <run>/preprocess/alignment_manifest.json --output-dir <alignment-figure>
```

Start with: `raw coordinates, spatial_ot_input, package-aligned coordinates, and alignment records`

Writes: `alignment comparison PDF/PNG files`

Next: `exact manuscript heart_extend_1/2 page builders`

The alignment calculations are available, but the code that assembled the current S7-S8 PNG pages has not been identified.

### 3. calculate growth (S9)
```text
cytobridge workflow --config chicken_heart --step downstream --aligned-h5ad <run>/preprocess/chicken_heart_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Start with: `aligned H5AD and retrained model used for the paper`

Writes: `<run>/downstream/growth/growth_by_cell.csv and growth_timepoint_grid.pdf`

Next: `exact manuscript heart_extend_3_revised_growth.png builder`

The growth table and plot are reproducible; the code that assembled the current S9 PNG page has not been identified.

### 4. calculate velocity components (S10)
```text
cytobridge workflow --config chicken_heart --step downstream --aligned-h5ad <run>/preprocess/chicken_heart_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Start with: `aligned H5AD and retrained model used for the paper`

Writes: `<run>/downstream/velocity/velocity_components.npz and full/drift/interaction vector PDFs`

Next: `exact manuscript heart_extend_4_revised_velocity.png builder`

The velocity arrays and plots are reproducible; the code that assembled the current S10 PNG and its veloAgent input have not been identified.

## Saved files

- Aligned data: `tutorial_outputs/chicken_heart/preprocess/chicken_heart_aligned.h5ad`
- Training directory: `tutorial_outputs/chicken_heart/training`
- Downstream directory: `tutorial_outputs/chicken_heart/downstream`